# 05 - Machine Learning, MLflow y Model Registry

Este notebook construye un dataset Gold para Machine Learning, entrena modelos de clasificación, registra métricas en MLflow y registra el mejor modelo en Unity Catalog.

## Problema

Clasificar si un accidente presenta víctimas:

- `tiene_victimas = 0`: solo daños.
- `tiene_victimas = 1`: con heridos o muertos.

El objetivo es una clasificación al momento conceptual del registro del accidente, no una predicción previa a que el accidente ocurra.

## Prevención de data leakage

No se utilizan como features `gravedad`, `nro_heridos`, `numero_muertos` ni información de la tabla de víctimas, porque revelan directa o indirectamente la variable objetivo.


## 1. Configuración


In [ ]:
from pathlib import Path
import sys

# Ejecutar desde el Git Folder completo, no desde un notebook importado aislado.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "accidentalidad" / "policies.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("No se encontró el paquete accidentalidad; abra el repositorio completo en un Git Folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from accidentalidad.policies import sql_identifier, parse_bool, nonnegative_int

import uuid
from pyspark.sql import functions as F
from accidentalidad.quality import QualityGate
from accidentalidad.policies import choose_candidate, promotion_allowed, temporal_partition

dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catálogo")
dbutils.widgets.dropdown("promote_champion", "false", ["false", "true"], "Evaluar promoción a Champion")
dbutils.widgets.text("minimum_validation_f1", "0.5", "F1 mínimo de validación")
dbutils.widgets.text("minimum_improvement", "0.0", "Mejora mínima sobre Champion")
dbutils.widgets.text("max_unknown_labels", "0", "Tolerancia de etiquetas desconocidas")
dbutils.widgets.text("max_driver_rows", "100000", "Límite antes de toPandas")
PROY_CATALOG = sql_identifier(dbutils.widgets.get("catalog_name"))
PROMOTE = parse_bool(dbutils.widgets.get("promote_champion"))
MIN_F1 = float(dbutils.widgets.get("minimum_validation_f1"))
MIN_IMPROVEMENT = float(dbutils.widgets.get("minimum_improvement"))
# Validar configuración incluso cuando no se solicita promoción.
promotion_allowed(0.5, 0.5, minimum=MIN_F1, improvement=MIN_IMPROVEMENT)
MAX_UNKNOWN = nonnegative_int(dbutils.widgets.get("max_unknown_labels"))
MAX_DRIVER_ROWS = nonnegative_int(dbutils.widgets.get("max_driver_rows"))
if MAX_DRIVER_ROWS == 0:
    raise ValueError("max_driver_rows debe ser positivo")
RUN_ID = uuid.uuid4().hex
SOURCE_TABLE = f"{PROY_CATALOG}.silver.accidentes"
FEATURE_TABLE = f"{PROY_CATALOG}.gold.features_accidentalidad"
MODEL_NAME = f"{PROY_CATALOG}.gold.modelo_riesgo_victimas"

## 2. Construcción de `gold.features_accidentalidad`

Se mantiene una fila por accidente (`croquis`). Las variables se construyen únicamente con información contextual disponible en el registro del accidente.


In [ ]:
SOURCE_VERSION = spark.sql(f"DESCRIBE HISTORY {SOURCE_TABLE}").select("version").first()["version"]
df_accidentes = spark.read.option("versionAsOf", SOURCE_VERSION).table(SOURCE_TABLE)
hora_texto = F.trim(F.col("hora_accidente"))
hora = F.expr("try_cast(split(trim(hora_accidente), ':')[0] AS INT)")
# También se valida la cadena completa: no aceptar 12abc o minutos/segundos inválidos.
hora_valida = hora_texto.rlike(r"^([01]?\d|2[0-3]):[0-5]\d(:[0-5]\d)?$")
invalid_hours = df_accidentes.filter(hora_texto.isNotNull() & (hora_texto != "") & ~hora_valida).count()
df_candidates = df_accidentes.select(
    "croquis", "anio_accidente", "mes_accidente",
    F.dayofweek("fecha_accidente").alias("dia_semana"),
    F.when(hora_valida, hora).otherwise(F.lit(None).cast("int")).alias("hora"),
    "localidad", "clase_accidente", "coordenada_x", "coordenada_y",
    F.when(F.col("gravedad") == "SOLO DAÑOS", 0)
     .when(F.col("gravedad").isin("CON HERIDOS", "CON MUERTOS"), 1)
     .otherwise(F.lit(None)).cast("int").alias("tiene_victimas"),
).withColumn("_silver_version", F.lit(SOURCE_VERSION)).withColumn("_quality_run_id", F.lit(RUN_ID))
unknown = df_candidates.filter(F.col("tiene_victimas").isNull())
unknown_count = unknown.count()
if unknown_count:
    (unknown.write.format("delta").mode("append")
     .saveAsTable(f"{PROY_CATALOG}.quality.unknown_labels"))
gate = QualityGate(spark, PROY_CATALOG, RUN_ID, "gold")
gate.check("features", "invalid_hour", invalid_hours)
gate.check("features", "unknown_label_limit", unknown_count <= MAX_UNKNOWN, True,
           details={"excluded": unknown_count, "limit": MAX_UNKNOWN})
if unknown_count:
    gate.check("features", "excluded_unknown_labels", unknown_count, severity="WARN")
df_features = df_candidates.filter(F.col("tiene_victimas").isNotNull())
gate.check("features", "empty_dataset", df_features.count() == 0, False)
gate.check("features", "null_croquis", df_features.filter(F.col("croquis").isNull()).count())
gate.check("features", "duplicate_croquis", df_features.groupBy("croquis").count().filter(F.col("count") > 1).count())
gate.check("features", "null_year", df_features.filter(F.col("anio_accidente").isNull()).count())
gate.finish()
(df_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FEATURE_TABLE))
FEATURE_VERSION = spark.sql(f"DESCRIBE HISTORY {FEATURE_TABLE}").select("version").first()["version"]
print(f"[OK] Gold v{FEATURE_VERSION}; fuente Silver v{SOURCE_VERSION}; excluidas={unknown_count}")

## 3. Validación de la variable objetivo


In [ ]:
df_target = (
    spark.table(FEATURE_TABLE)
    .groupBy("tiene_victimas")
    .count()
)

total_registros = spark.table(FEATURE_TABLE).count()

df_target = (
    df_target
    .withColumn(
        "porcentaje",
        F.round(F.col("count") / F.lit(total_registros) * 100, 2)
    )
    .orderBy("tiene_victimas")
)

display(df_target)

## 4. Tres particiones temporales
- Entrenamiento: 2018–2020.
- Validación: 2021, usada para elegir candidato y decidir promoción.
- Prueba final: 2022, evaluada únicamente para el candidato elegido.
Se exigen ambas clases en cada partición. Otros años se excluyen de ML y se contabilizan.
La comparación pierde validez si se ajustan repetidamente decisiones observando el test;
para nuevas iteraciones se requiere un nuevo periodo de prueba.

In [ ]:
fixed_features = spark.read.option("versionAsOf", FEATURE_VERSION).table(FEATURE_TABLE)
row_count = fixed_features.count()
if row_count > MAX_DRIVER_ROWS:
    raise ValueError(f"{row_count} filas superan max_driver_rows={MAX_DRIVER_ROWS}; use entrenamiento distribuido o aumente el límite tras dimensionar memoria")
df_ml = fixed_features.orderBy("croquis").toPandas()
feature_columns = ["mes_accidente", "dia_semana", "hora", "localidad", "clase_accidente", "coordenada_x", "coordenada_y"]
# Convertir nulos Spark a NaN de forma consistente con SimpleImputer.
import numpy as np
import pandas as pd
df_ml[feature_columns] = df_ml[feature_columns].replace({None: np.nan})
# Una firma DOUBLE permite representar NaN; una firma LONG impediría imputar nulos en serving.
numeric_columns = ["mes_accidente", "dia_semana", "hora", "coordenada_x", "coordenada_y"]
df_ml[numeric_columns] = df_ml[numeric_columns].astype("float64")
partitions = df_ml["anio_accidente"].map(temporal_partition)
train_df = df_ml[partitions == "train"].copy()
validation_df = df_ml[partitions == "validation"].copy()
test_df = df_ml[partitions == "test"].copy()
excluded_period_rows = int((partitions == "excluded").sum())
for name, frame in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
    if frame.empty or set(frame["tiene_victimas"].unique()) != {0, 1}:
        raise ValueError(f"{name}: se requieren filas y ambas clases")
    print(f"{name}: {len(frame)} filas, tasa positiva={frame['tiene_victimas'].mean():.4f}")
X_train, y_train = train_df[feature_columns], train_df["tiene_victimas"].astype(int)
X_validation, y_validation = validation_df[feature_columns], validation_df["tiene_victimas"].astype(int)
X_test, y_test = test_df[feature_columns], test_df["tiene_victimas"].astype(int)
print(f"Filas fuera de 2018-2022 excluidas del entrenamiento/evaluación: {excluded_period_rows}")

## 5. Pipeline de preprocesamiento


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = ["mes_accidente", "dia_semana", "hora", "coordenada_x", "coordenada_y"]
categorical_features = ["localidad", "clase_accidente"]
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

## 6. Entrenamiento y selección con validación
Se compara Regresión Logística y Random Forest contra un baseline de clase mayoritaria.
Cada candidato registra hiperparámetros, métricas de validación, configuración, versiones
exactas de dependencias, hashes del código, commit cuando está disponible y versiones Delta.
El preprocesador se ajusta solo con entrenamiento y se clona por candidato.

In [ ]:
import hashlib
import inspect
import json
import subprocess
from importlib.metadata import version
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, classification_report

current_user = spark.sql("SELECT current_user() AS user").first()["user"]
EXPERIMENT_NAME = f"/Users/{current_user}/accidentalidad_databricks_ml"
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

packages = ["mlflow", "scikit-learn", "pandas", "numpy", "scipy", "cloudpickle", "databricks-sdk"]
environment = {package: version(package) for package in packages}
model_requirements = [f"{package}=={environment[package]}" for package in packages if package != "databricks-sdk"]
files = sorted([*PROJECT_ROOT.glob("accidentalidad/*.py"), *PROJECT_ROOT.glob("*/*.ipynb")])
code_manifest = {str(path.relative_to(PROJECT_ROOT)): hashlib.sha256(path.read_bytes()).hexdigest() for path in files}
code_sha256 = hashlib.sha256(json.dumps(code_manifest, sort_keys=True).encode()).hexdigest()
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True, stderr=subprocess.DEVNULL).strip()
    dirty = bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=PROJECT_ROOT, text=True, stderr=subprocess.DEVNULL).strip())
except (OSError, subprocess.CalledProcessError):
    commit, dirty = "unavailable", "unknown"
provenance = {
    "source_table": SOURCE_TABLE, "source_version": int(SOURCE_VERSION),
    "feature_table": FEATURE_TABLE, "feature_version": int(FEATURE_VERSION),
    "train_period": "2018-2020", "validation_period": "2021", "test_period": "2022",
    "train_end_year": 2020, "quality_run_id": RUN_ID, "git_commit": commit, "git_dirty": str(dirty),
    "code_sha256": code_sha256, "excluded_unknown_labels": unknown_count,
    "excluded_period_rows": excluded_period_rows,
}

def evaluate(model, X, y):
    pred = model.predict(X)
    # Las etiquetas están validadas como 0/1; ubicar explícitamente la clase positiva.
    classes = list(model.classes_)
    prob = model.predict_proba(X)[:, classes.index(1)]
    return {
        "accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0), "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, prob), "pr_auc": average_precision_score(y, prob),
    }, classification_report(y, pred, output_dict=True, zero_division=0)

# Baseline aprende la clase más frecuente de entrenamiento, sin usar el test.
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_metrics, _ = evaluate(baseline, X_validation, y_validation)
BASELINE_F1 = baseline_metrics["f1"]
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1),
}
results = []
for model_name, estimator in models.items():
    # Cada candidato tiene su propio preprocesador; no comparte estado ajustado.
    pipeline = Pipeline([("preprocessor", clone(preprocessor)), ("classifier", estimator)])
    with mlflow.start_run(run_name=model_name) as run:
        pipeline.fit(X_train, y_train)
        metrics, report = evaluate(pipeline, X_validation, y_validation)
        mlflow.log_params({**provenance, "model_name": model_name, "feature_count": len(feature_columns),
                           "train_rows": len(X_train), "validation_rows": len(X_validation), "test_rows": len(X_test)})
        mlflow.log_params({f"classifier_{key}": value for key, value in estimator.get_params().items()})
        mlflow.log_dict(environment, "environment_versions.json")
        mlflow.log_dict(code_manifest, "code_manifest.json")
        mlflow.log_dict({"features": feature_columns, "preprocessor": repr(preprocessor), "estimator": estimator.get_params()}, "training_configuration.json")
        mlflow.log_metrics({f"validation_{key}": value for key, value in metrics.items()})
        mlflow.log_metric("baseline_validation_f1", BASELINE_F1)
        mlflow.log_dict(report, "validation_classification_report.json")
        kwargs = {"sk_model": pipeline, "signature": infer_signature(X_train, pipeline.predict(X_train)),
                  "input_example": X_train.head(5), "pip_requirements": model_requirements}
        kwargs["name" if "name" in inspect.signature(mlflow.sklearn.log_model).parameters else "artifact_path"] = "model"
        model_info = mlflow.sklearn.log_model(**kwargs)
        results.append({"model": model_name, "run_id": run.info.run_id,
                        "model_uri": model_info.model_uri, **{f"validation_{key}": value for key, value in metrics.items()}})
results_df = pd.DataFrame(results).sort_values(["validation_f1", "model"], ascending=[False, True])
display(results_df)

## 7. Prueba final y registro del candidato
La prueba de 2022 se evalúa únicamente para el ganador de validación. El modelo registrado
conserva su entrenamiento 2018–2020; no se reajusta con validación. Registrar una versión
no cambia el alias Champion.

In [ ]:
best = choose_candidate(results)
selected_model = mlflow.sklearn.load_model(best["model_uri"])
# El test se abre después de elegir el candidato, no durante la comparación.
test_metrics, test_report = evaluate(selected_model, X_test, y_test)
with mlflow.start_run(run_id=best["run_id"]):
    mlflow.log_metrics({f"test_{key}": value for key, value in test_metrics.items()})
    mlflow.log_dict(test_report, "test_classification_report.json")
    mlflow.set_tag("selected_by", "validation_f1_2021")
registered_model = mlflow.register_model(model_uri=best["model_uri"], name=MODEL_NAME)
from mlflow import MlflowClient
client = MlflowClient(registry_uri="databricks-uc")
for key, value in {**provenance, "validation_f1": best["validation_f1"], "selection_run_id": best["run_id"]}.items():
    client.set_model_version_tag(MODEL_NAME, registered_model.version, key, str(value))
print(f"[OK] Candidato registrado: {MODEL_NAME}, versión {registered_model.version}")
print(f"Validación F1={best['validation_f1']:.4f}; prueba F1={test_metrics['f1']:.4f}")

# Comprobar la serialización antes de permitir cualquier modificación de Champion.
candidate_uri = f"models:/{MODEL_NAME}/{registered_model.version}"
candidate_model = mlflow.pyfunc.load_model(candidate_uri)
sample = X_test.head(5).copy()
expected = selected_model.predict(sample)
actual = candidate_model.predict(sample)
if not np.array_equal(expected, actual):
    raise ValueError("La inferencia del candidato registrado difiere del modelo seleccionado")


## 8. Promoción explícita
`promote_champion=true` exige F1 de validación mínimo, mejora estricta sobre baseline y,
si existe, mejora sobre Champion reevaluado en la misma validación. El Champion debe tener
evidencia de entrenamiento anterior a 2021. Las métricas de prueba no deciden la promoción.
Ejecute una sola promoción a la vez; la API de alias no proporciona aquí un bloqueo concurrente.

In [ ]:
PROMOTED = False
if PROMOTE:
    from mlflow.exceptions import MlflowException
    champion = None
    try:
        champion = client.get_model_version_by_alias(MODEL_NAME, "Champion")
    except MlflowException as error:
        # Permisos, conexión y demás errores no se interpretan como ausencia de alias.
        if error.error_code != "RESOURCE_DOES_NOT_EXIST":
            raise
    champion_f1 = None
    if champion is not None:
        tags = champion.tags or {}
        if not tags.get("train_end_year") or int(tags["train_end_year"]) >= 2021:
            raise ValueError("Champion no tiene evidencia de entrenamiento anterior a 2021; no es posible compararlo en esa validación")
        current_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{champion.version}")
        champion_metrics, _ = evaluate(current_model, X_validation, y_validation)
        champion_f1 = champion_metrics["f1"]
    PROMOTED = promotion_allowed(best["validation_f1"], BASELINE_F1, champion_f1, MIN_F1, MIN_IMPROVEMENT)
    with mlflow.start_run(run_id=best["run_id"]):
        mlflow.log_params({"promotion_minimum_f1": MIN_F1, "promotion_minimum_improvement": MIN_IMPROVEMENT})
        if champion_f1 is not None:
            mlflow.log_metric("incumbent_validation_f1", champion_f1)
        mlflow.set_tag("promotion_eligible", str(PROMOTED))
    if not PROMOTED:
        raise ValueError("Promoción rechazada: el candidato debe alcanzar el mínimo y superar baseline y Champion en validación")
    client.set_registered_model_alias(MODEL_NAME, "Champion", registered_model.version)
    print(f"[OK] Champion -> {registered_model.version}")
else:
    print("Candidato registrado; promote_champion=false")

## 9. Inferencia desde Model Registry
Se compara la inferencia del modelo registrado con el seleccionado. Si hubo promoción,
también se verifica la carga por alias Champion.

In [ ]:
# La igualdad del candidato ya se comprobó antes de la posible promoción.
if PROMOTED:
    champion_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@Champion")
    if not np.array_equal(actual, champion_model.predict(sample)):
        raise ValueError("La inferencia mediante Champion no coincide con el candidato promovido")
sample["prediccion_tiene_victimas"] = actual
display(sample)

## 10. Evidencia y límites
Gold, resultados de calidad, experimentos y versiones del modelo se generan al ejecutar en
Databricks. El repositorio no contiene métricas de ejecución certificadas. La clasificación
usa información contextual del accidente; confirme la disponibilidad de `clase_accidente`
y coordenadas en el instante de uso antes de emplearla como predicción operativa.